In [9]:
import pandas as pd
import numpy as np

# ================================
# CONFIG
# ================================
panel_name = "Aging-SFO_C2P2329.csv"
panel_path = f"panels/{panel_name}"

combined_path = "score_panels/combined_panel_sorted.csv"

sample_size = 1000        # K: numero di geni campionati con rimpiazzo
threshold_fraction = 0.10 # 0.70 = 70%, 1.0 = 100%, ecc.


# ================================
# STEP 1 — LOAD PANEL
# ================================
panel_df = pd.read_csv(panel_path)
panel_df["Ensembl_ID"] = panel_df["Ensembl_ID"].astype(str)
panel_genes = set(panel_df["Ensembl_ID"])

panel_size = len(panel_genes)
k_min = int(np.ceil(threshold_fraction * panel_size))

print("\n=== PANEL INFO ===")
print(f"Panel: {panel_name}")
print(f"Panel size: {panel_size}")
print(f"Threshold (>= {threshold_fraction*100:.1f}%): {k_min} genes")


# ================================
# STEP 2 — LOAD PROBABILITIES
# ================================
combined = pd.read_csv(combined_path)
combined["Ensembl_ID"] = combined["Ensembl_ID"].astype(str)

if "Combined_score_prob" not in combined.columns:
    raise ValueError("Column 'Combined_score_prob' not found in combined_panel.csv")

# dizionario gene → p
p_dict = dict(zip(combined["Ensembl_ID"], combined["Combined_score_prob"]))

# controlliamo quali geni del pannello hanno una probabilità valida
panel_probs_list = []
missing_genes = []

for g in panel_genes:
    if g in p_dict:
        panel_probs_list.append(p_dict[g])
    else:
        missing_genes.append(g)

panel_probs = np.array(panel_probs_list)

print("\n=== PROB INFO ===")
print(f"Original panel size: {panel_size}")
print(f"Panel genes missing from combined_panel.csv: {len(missing_genes)}")

if len(missing_genes) > 0:
    print("These genes will be ignored in probability calculations.")
    # se vuoi vedere i nomi, scommenta:
    # print("Missing genes:", missing_genes)

# aggiorniamo il panel_size (solo geni con probabilità)
panel_size_valid = len(panel_probs)

# aggiorniamo la soglia basata sui geni validi
threshold_hits = int(np.ceil(threshold_fraction * panel_size_valid))

print(f"Valid panel size (with prob): {panel_size_valid}")
print(f"Threshold (>= {threshold_fraction*100:.1f}%): {threshold_hits} genes")

# somma totale
print("Sum of probabilities over valid panel genes:", panel_probs.sum())


# ================================
# STEP 3 — INCLUSION PROB q_g = P(gene appears at least once)
# ================================
# con rimpiazzo: q_g = 1 - (1 - p_g)^K
q = 1.0 - np.power(1.0 - panel_probs, sample_size)

print("\nExample q_g (first 5):", q[:5])
print("Sum of q_g (expected distinct panel genes) ≈", q.sum())


# ================================
# STEP 4 — POISSON–BINOMIAL PMF for X = #distinct panel genes
# ================================
def poisson_binomial_pmf(p_vec):
    """
    p_vec: array di probabilità q_g
    ritorna pmf[k] = P(X = k), X ~ Poisson-Binomial
    """
    n = len(p_vec)
    pmf = np.zeros(n + 1)
    pmf[0] = 1.0

    for prob in p_vec:
        pmf[1:] = pmf[1:] * (1 - prob) + pmf[:-1] * prob
        pmf[0] *= (1 - prob)

    return pmf

pmf = poisson_binomial_pmf(q)


# ================================
# STEP 5 — PROBABILITY P(X >= k_min)
# ================================
prob_atleast = pmf[k_min:].sum()

# X = panel_size_valid → tutti i geni validi del pannello appaiono almeno una volta
prob_all = pmf[panel_size_valid]

print("\n==================== RESULT (WITH REPLACEMENT MODEL) ====================")
print(f"P( >= {threshold_fraction*100:.1f}% of {panel_name} panel )  ≈ {prob_atleast:.6e}")
print(f"P( 100% of Allen_Zeng panel )           ≈ {prob_all:.6e}")
print("=======================================================================\n")



=== PANEL INFO ===
Panel: Aging-SFO_C2P2329.csv
Panel size: 501
Threshold (>= 10.0%): 51 genes

=== PROB INFO ===
Original panel size: 501
Panel genes missing from combined_panel.csv: 2
These genes will be ignored in probability calculations.
Valid panel size (with prob): 499
Threshold (>= 10.0%): 50 genes
Sum of probabilities over valid panel genes: 0.024750815670729627

Example q_g (first 5): [0.04515427 0.0583168  0.04248364 0.04809695 0.03014741]
Sum of q_g (expected distinct panel genes) ≈ 24.068947659561942

==================== RESULT (WITH REPLACEMENT MODEL) ====================
P( >= 10.0% of Aging-SFO_C2P2329.csv panel )  ≈ 4.964115e-07
P( 100% of Allen_Zeng panel )           ≈ 0.000000e+00



In [10]:
import os
import pandas as pd

PANEL_FOLDER = "panels"
SCORE_FOLDER = "score_panels"

# -----------------------------------------------------
# 1. Carica i pannelli reali e conta la frequenza dei geni
# -----------------------------------------------------

gene_panel_counts = {}   # gene → # pannelli reali in cui appare
all_real_genes = set()

for fname in os.listdir(PANEL_FOLDER):
    if fname.endswith(".csv"):
        df = pd.read_csv(os.path.join(PANEL_FOLDER, fname))
        col = df.columns[0]
        genes = set(df[col].astype(str))

        for g in genes:
            gene_panel_counts[g] = gene_panel_counts.get(g, 0) + 1

        all_real_genes |= genes  # unione di tutti i geni dei pannelli reali

print(f"\nTotale geni presenti in almeno 1 pannello reale: {len(all_real_genes)}\n")


# -----------------------------------------------------
# 2. Carica tutti i ranking (score_panels)
# rank = numero di riga (partendo da 1)
# -----------------------------------------------------

score_files = [f for f in os.listdir(SCORE_FOLDER) if f.endswith(".csv")]
score_tables = {}   # file → dict gene → rank

for fname in score_files:
    df = pd.read_csv(os.path.join(SCORE_FOLDER, fname))
    col = df.columns[0]

    mapping = {gene: i for i, gene in enumerate(df[col].astype(str), start=1)}
    score_tables[fname] = mapping

print("Trovati ranking:", score_files, "\n")


# -----------------------------------------------------
# 3. Per ogni gene reale → trova posizione in ogni score panel
# -----------------------------------------------------

gene_ranks = {}   # gene → {file: rank, ..., "panel_count": n}

for gene in all_real_genes:
    gene_info = {"panel_count": gene_panel_counts[gene]}

    for fname in score_files:
        rank = score_tables[fname].get(gene, None)
        gene_info[fname] = rank

    gene_ranks[gene] = gene_info


# -----------------------------------------------------
# 4. Conta quanti geni sono nei top K in almeno un ranking
# -----------------------------------------------------

thresholds = [100, 200, 400, 800, 1600, 3200]
counts = {k: 0 for k in thresholds}

for gene, info in gene_ranks.items():
    ranks = [info[fname] for fname in score_files if info[fname] is not None]

    for k in thresholds:
        if any(r <= k for r in ranks):
            counts[k] += 1

# -----------------------------------------------------
# 5. Print dei risultati
# -----------------------------------------------------

print("\n===================== RISULTATI =====================")

print("\nEsempio di output per alcuni geni:\n")
for i, (gene, info) in enumerate(list(gene_ranks.items())[:10]):
    print(f"GENE: {gene}")
    print(f"  Present in {info['panel_count']} real panels")
    for fname in score_files:
        print(f"   {fname:35s} rank = {info[fname]}")
    print()

print("\n================== TOP-K SUMMARY ==================\n")
for k in thresholds:
    print(f"Geni nei top {k:4d} in almeno 1 ranking: {counts[k]}")

print("\n====================================================\n")



Totale geni presenti in almeno 1 pannello reale: 2994

Trovati ranking: ['PCA_genes_full_panel.csv', 'HVG_panel.csv', 'combined_panel_full_sorted.csv', 'combined_panel_sorted.csv', 'DE_panel_leiden_0.6.csv'] 


===================== RISULTATI =====================

Esempio di output per alcuni geni:

GENE: ENSMUSG00000018604
  Present in 4 real panels
   PCA_genes_full_panel.csv            rank = 4955
   HVG_panel.csv                       rank = 1007
   combined_panel_full_sorted.csv      rank = 2401
   combined_panel_sorted.csv           rank = 2401
   DE_panel_leiden_0.6.csv             rank = 8717

GENE: ENSMUSG00000043668
  Present in 3 real panels
   PCA_genes_full_panel.csv            rank = 2551
   HVG_panel.csv                       rank = 19531
   combined_panel_full_sorted.csv      rank = 5585
   combined_panel_sorted.csv           rank = 5585
   DE_panel_leiden_0.6.csv             rank = 14745

GENE: ENSMUSG00000023885
  Present in 1 real panels
   PCA_genes_full_panel.csv